# Experiment 1: Generate Random Samples from Various Probability Distributions using Python

## Objective
Generate random samples from key probability distributions — **Normal, Uniform, Binomial, and Poisson** — and understand their relevance in **Generative AI (GenAI)**.

## Why Probability Distributions Matter in Generative AI

Generative AI models are fundamentally built on probability. Whether generating text, images, audio, or code, every GenAI model is essentially **learning and sampling from complex probability distributions**:

| Distribution | GenAI Application |
|---|---|
| **Normal (Gaussian)** | Latent space in VAEs & GANs; weight initialization in neural networks |
| **Uniform** | Random seed generation; dropout regularization; random token sampling |
| **Binomial** | Bernoulli dropout; binary classification in discriminators (GAN) |
| **Poisson** | Modelling rare token occurrences; event-driven generation |

> **Key Insight:** When a LLM like GPT or Claude generates text, it samples the next token from a learned probability distribution over the vocabulary. Temperature scaling shifts how 'peaked' or 'spread out' that distribution is.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import pandas as pd

# Reproducibility
np.random.seed(42)

# Style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

N = 1000  # Sample size
print(f'Setup complete. Generating {N} samples per distribution.')

---
## 1. Normal (Gaussian) Distribution

### Theory
The **Normal distribution** N(mu, sigma^2) is the cornerstone of Generative AI. It is symmetric and bell-shaped, described by:
- **Mean** mu: centre of the distribution
- **Std Dev** sigma: spread / uncertainty

### GenAI Connection
- **Variational Autoencoders (VAEs):** The encoder maps inputs to a Gaussian latent space z ~ N(mu, sigma^2). Sampling from this space lets the decoder *generate* new data.
- **GAN weight initialisation:** Weights are drawn from N(0, 0.02) so gradients flow well at the start of training.
- **Diffusion Models (DALL-E 3, Stable Diffusion):** Start with pure Gaussian noise x_T ~ N(0, I) and iteratively denoise to generate images.

In [ ]:
# 1. Normal Distribution
mu, sigma = 0, 1
samples_normal = np.random.normal(loc=mu, scale=sigma, size=N)

print('=== Normal Distribution - Summary Statistics ===')
print(f'  Mean      : {np.mean(samples_normal):.4f}  (theoretical: {mu})')
print(f'  Std Dev   : {np.std(samples_normal):.4f}  (theoretical: {sigma})')
print(f'  Skewness  : {stats.skew(samples_normal):.4f}  (theoretical: 0)')
print(f'  Kurtosis  : {stats.kurtosis(samples_normal):.4f}  (theoretical: 0)')
print(f'  Min / Max : {samples_normal.min():.3f} / {samples_normal.max():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram + KDE
sns.histplot(samples_normal, bins=35, kde=True, color='steelblue', ax=axes[0])
x_range = np.linspace(-4, 4, 300)
axes[0].plot(x_range, stats.norm.pdf(x_range, mu, sigma) * N * (8/35),
             'r--', linewidth=2, label='Theoretical PDF')
axes[0].set_title('Normal Distribution - Histogram & KDE\nN(mean=0, std=1)', fontsize=12)
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')
axes[0].axvline(mu, color='orange', linestyle=':', linewidth=2, label='Mean')
axes[0].legend()

# Q-Q Plot
stats.probplot(samples_normal, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot (Normal)\nPoints on line = good Gaussian fit', fontsize=12)

plt.suptitle('GenAI Link: VAE latent space & Diffusion Model noise prior',
             fontsize=11, style='italic', color='dimgray', y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Uniform Distribution

### Theory
Every value in the range [a, b] is equally likely. Key properties:
- **Mean** = (a + b) / 2
- **Variance** = (b - a)^2 / 12

### GenAI Connection
- **Random noise injection:** Uniform noise U(0,1) is used in **dropout** regularisation — a neuron is kept with probability p, dropped with 1-p, preventing overfitting in generative models.
- **Temperature sampling:** Uniform random numbers feed into the inverse-CDF trick to sample tokens from a model's vocabulary distribution without bias.
- **GAN training:** The discriminator is initialised with uniform label noise to stabilise early training.

In [ ]:
# 2. Uniform Distribution
low, high = 0, 1
samples_uniform = np.random.uniform(low=low, high=high, size=N)

print('=== Uniform Distribution - Summary Statistics ===')
print(f'  Mean      : {np.mean(samples_uniform):.4f}  (theoretical: {(low+high)/2})')
print(f'  Std Dev   : {np.std(samples_uniform):.4f}  (theoretical: {np.sqrt((high-low)**2/12):.4f})')
print(f'  Min / Max : {samples_uniform.min():.4f} / {samples_uniform.max():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(samples_uniform, bins=30, kde=True, color='coral', ax=axes[0])
axes[0].axhline(y=N/30, color='navy', linestyle='--', linewidth=2, label='Theoretical density')
axes[0].set_title('Uniform Distribution - Histogram & KDE\nU(a=0, b=1)', fontsize=12)
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Empirical CDF vs theoretical
sorted_s = np.sort(samples_uniform)
ecdf = np.arange(1, N+1) / N
axes[1].plot(sorted_s, ecdf, color='coral', linewidth=2, label='Empirical CDF')
axes[1].plot([0, 1], [0, 1], 'navy--', linewidth=2, label='Theoretical CDF')
axes[1].set_title('Empirical vs Theoretical CDF\nFlat histogram = uniform coverage', fontsize=12)
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Cumulative Probability')
axes[1].legend()

plt.suptitle('GenAI Link: Dropout regularisation & inverse-CDF token sampling',
             fontsize=11, style='italic', color='dimgray', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Binomial Distribution

### Theory
Counts the number of **successes** in n independent Bernoulli trials each with probability p:
- **Mean** = n * p
- **Variance** = n * p * (1 - p)

### GenAI Connection
- **Bernoulli Dropout:** Each neuron is independently dropped (Bernoulli trial). With n neurons and drop probability p, active neurons follow Binomial(n, 1-p).
- **GAN Discriminator:** Outputs a binary decision - real or fake. Training the discriminator is a Binomial likelihood optimisation.
- **Token masking in BERT / masked language models:** k out of n tokens are randomly masked following a Binomial process.

In [ ]:
# 3. Binomial Distribution
n_trials, p_success = 10, 0.5
samples_binomial = np.random.binomial(n=n_trials, p=p_success, size=N)

print('=== Binomial Distribution - Summary Statistics ===')
print(f'  Mean      : {np.mean(samples_binomial):.4f}  (theoretical: {n_trials * p_success})')
print(f'  Variance  : {np.var(samples_binomial):.4f}  (theoretical: {n_trials * p_success * (1-p_success)})')
print(f'  Std Dev   : {np.std(samples_binomial):.4f}')
print(f'  Min / Max : {samples_binomial.min()} / {samples_binomial.max()}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

k_values = np.arange(0, n_trials + 1)
empirical_freq = [np.sum(samples_binomial == k) for k in k_values]
theoretical_prob = stats.binom.pmf(k_values, n_trials, p_success) * N

axes[0].bar(k_values - 0.2, empirical_freq, width=0.4, color='mediumseagreen', label='Empirical')
axes[0].bar(k_values + 0.2, theoretical_prob, width=0.4, color='navy', alpha=0.6, label='Theoretical')
axes[0].set_title('Binomial Distribution - Empirical vs Theoretical PMF\nB(n=10, p=0.5)', fontsize=12)
axes[0].set_xlabel('Number of Successes (k)')
axes[0].set_ylabel('Count')
axes[0].set_xticks(k_values)
axes[0].legend()

axes[1].boxplot(samples_binomial, vert=True, patch_artist=True,
                boxprops=dict(facecolor='mediumseagreen', color='darkgreen'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Box Plot - Spread of Binomial Samples\nMedian close to mean = symmetric', fontsize=12)
axes[1].set_ylabel('Number of Successes')
axes[1].set_xticks([])

plt.suptitle('GenAI Link: Bernoulli Dropout & GAN discriminator training',
             fontsize=11, style='italic', color='dimgray', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Poisson Distribution

### Theory
Models the number of **rare events** in a fixed interval given average rate lambda:
- **Mean** = lambda
- **Variance** = lambda (equal to mean - a unique property of Poisson)

### GenAI Connection
- **Rare token modelling:** In language models, low-frequency (rare) words appear according to a Poisson-like process. Handling rare tokens is critical for diverse text generation.
- **Data augmentation scheduling:** Poisson processes model random augmentation event intervals in training pipelines.
- **Neural event models:** Recurrent generative models for sequences of discrete events (music note generation, code commit prediction) often use Poisson assumptions.

In [ ]:
# 4. Poisson Distribution
lambda_param = 3
samples_poisson = np.random.poisson(lam=lambda_param, size=N)

print('=== Poisson Distribution - Summary Statistics ===')
print(f'  Mean      : {np.mean(samples_poisson):.4f}  (theoretical: {lambda_param})')
print(f'  Variance  : {np.var(samples_poisson):.4f}  (theoretical: {lambda_param})')
print(f'  Std Dev   : {np.std(samples_poisson):.4f}  (theoretical: {np.sqrt(lambda_param):.4f})')
print(f'  Var/Mean  : {np.var(samples_poisson)/np.mean(samples_poisson):.4f}  (should be approx 1.0 for Poisson)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

max_k = samples_poisson.max()
k_values = np.arange(0, max_k + 1)
empirical_freq = [np.sum(samples_poisson == k) for k in k_values]
theoretical_prob = stats.poisson.pmf(k_values, lambda_param) * N

axes[0].bar(k_values - 0.2, empirical_freq, width=0.4, color='mediumpurple', label='Empirical')
axes[0].bar(k_values + 0.2, theoretical_prob, width=0.4, color='darkviolet', alpha=0.5, label='Theoretical')
axes[0].set_title('Poisson Distribution - Empirical vs Theoretical PMF\nPois(lambda=3)', fontsize=12)
axes[0].set_xlabel('Number of Events (k)')
axes[0].set_ylabel('Count')
axes[0].set_xticks(k_values)
axes[0].legend()

axes[1].bar(k_values, theoretical_prob, color='darkviolet', alpha=0.7, label='Theoretical')
axes[1].bar(k_values, empirical_freq, color='mediumpurple', alpha=0.5, label='Empirical')
axes[1].set_yscale('log')
axes[1].set_title('Poisson - Log-Scale (Tail Behaviour)\nRare events visible in the right tail', fontsize=12)
axes[1].set_xlabel('Number of Events (k)')
axes[1].set_ylabel('Count (log scale)')
axes[1].set_xticks(k_values)
axes[1].legend()

plt.suptitle('GenAI Link: Rare token modelling & event-based sequence generation',
             fontsize=11, style='italic', color='dimgray', y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Combined Comparison - All Four Distributions

In [ ]:
# 5. Combined Dashboard
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

configs = [
    (samples_normal,   'Normal N(0,1)',          'steelblue',    False),
    (samples_uniform,  'Uniform U(0,1)',          'coral',        False),
    (samples_binomial, 'Binomial B(10, 0.5)',     'mediumseagreen', True),
    (samples_poisson,  'Poisson (lambda=3)',      'mediumpurple', True),
]

for idx, (samples, title, color, discrete) in enumerate(configs):
    ax = fig.add_subplot(gs[idx // 2, idx % 2])
    if discrete:
        unique_vals = np.arange(samples.min(), samples.max() + 1)
        sns.histplot(samples, bins=len(unique_vals), discrete=True, color=color, ax=ax, kde=False)
    else:
        sns.histplot(samples, bins=35, kde=True, color=color, ax=ax)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    mean_val = np.mean(samples)
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.8, label=f'Mean={mean_val:.2f}')
    ax.legend(fontsize=9)

fig.suptitle('Experiment 1 - All Probability Distributions at a Glance\n(N=1000 samples each, seed=42)',
             fontsize=14, fontweight='bold', y=1.01)
plt.show()

---
## 6. Summary Table

In [ ]:
# 6. Summary Table
summary = {
    'Distribution'    : ['Normal', 'Uniform', 'Binomial', 'Poisson'],
    'Parameters'      : ['mu=0, sigma=1', 'a=0, b=1', 'n=10, p=0.5', 'lambda=3'],
    'Empirical Mean'  : [f'{np.mean(s):.4f}' for s in [samples_normal, samples_uniform, samples_binomial, samples_poisson]],
    'Theoretical Mean': ['0.0000', '0.5000', '5.0000', '3.0000'],
    'Empirical Std'   : [f'{np.std(s):.4f}' for s in [samples_normal, samples_uniform, samples_binomial, samples_poisson]],
    'Theoretical Std' : ['1.0000', '0.2887', '1.5811', '1.7321'],
    'Type'            : ['Continuous', 'Continuous', 'Discrete', 'Discrete'],
    'GenAI Use Case'  : [
        'VAE latent space, Diffusion noise prior',
        'Dropout, inverse-CDF token sampling',
        'Bernoulli dropout, BERT token masking',
        'Rare token modelling, event sequences'
    ]
}

df_summary = pd.DataFrame(summary).set_index('Distribution')
print('EXPERIMENT 1 - SUMMARY TABLE')
print('=' * 90)
print(df_summary.to_string())
print('=' * 90)

---
## Conclusion

In this experiment we:
1. Generated **1000 random samples** from Normal, Uniform, Binomial, and Poisson distributions.
2. Validated samples against **theoretical parameters** using mean, std, skewness, PMF/PDF curves, and Q-Q plots.
3. Connected each distribution to a **real Generative AI application**.

### Key Takeaways for Generative AI

| Insight | Implication |
|---|---|
| Normal distribution is the backbone of the GenAI latent space | Sampling from N(0,I) and decoding = generation |
| Uniform sampling enables unbiased token selection | Controls creativity via temperature |
| Binomial distribution governs regularisation | Dropout prevents memorisation in generators |
| Poisson distribution models rare linguistic events | Critical for long-tail vocabulary handling |

> **Next Steps:** Experiment 2 will explore how these distributions appear inside neural network weight tensors and how they evolve during training.